# Churn · signals, baselines, hand-in v1

Issue #8, first pass. Three questions: what in the customer table predicts churn, what do three plain models get on identical folds, and a first prediction file so the 14.10 hand-in is covered before we tune anything.

Reads `data/raw/` through `src.data`, writes to `results/` (gitignored).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src import style
from src.churn import (FEATURE_SETS, best_threshold, cross_validate, feature_matrix,
                       save_results, summary_row, train_predict_split, write_handin)
from src.data import load_customer_table
from src.features import build_customer_features

style.apply()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

In [ ]:
feats = build_customer_features(load_customer_table())
train = feats[feats["split"] == "train"].reset_index(drop=True)
y = train["churned"].astype(bool).astype(int).to_numpy()

print("train", train.shape, "| predict", (feats["split"] == "predict").sum(), "| churn rate", y.mean().round(3))

## what are the signals

churn rate by quintile of a few features. every feature is cut into 5 equal-sized bins on the train set, the bar is the churn rate inside the bin. `leak_share` has too many zeros for quintiles, fixed bins instead.

In [ ]:
def fmt(v):
    if v <= 0:
        return "0"
    if v >= 1000:
        return f"{v / 1000:.1f}k"
    if v >= 10:
        return f"{v:.0f}"
    return f"{v:.2g}"


def churn_by_bin(df, col, bins=None):
    b = pd.qcut(df[col], 5, duplicates="drop") if bins is None else pd.cut(df[col], bins)
    g = df.groupby(b, observed=True)["churned"].agg(rate="mean", n="size")
    g.index = [f"{fmt(iv.left)}–{fmt(iv.right)}" if iv.right > 0 else "0" for iv in g.index]
    return g

panels = [
    ("days_active", None), ("n_transactions", None), ("total_amount", None),
    ("n_country", None), ("foreign_tx_share", None), ("leak_share", [-0.01, 0, 0.05, 0.2, 0.5, 1]),
]
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, (col, bins) in zip(axes.flat, panels):
    g = churn_by_bin(train.assign(churned=y), col, bins)
    ax.bar(g.index, g["rate"] * 100, color=style.C[0], width=0.7)
    style.annotate_bars(ax, g["rate"] * 100, [f"n={n}" for n in g["n"]])
    style.clean(ax, col, "% churned")
    ax.set_ylim(0, 105)
    ax.tick_params(axis="x", labelsize=8, rotation=20)
plt.tight_layout()

In [ ]:
X = feature_matrix(train, "engineered")
corr = X.corrwith(pd.Series(y, index=X.index)).dropna().sort_values(key=np.abs, ascending=False)

fig, ax = plt.subplots(figsize=(7, 5))
top = corr.head(15)[::-1]
ax.barh(top.index, top.values, color=[style.C[1] if v > 0 else style.C[0] for v in top.values])
style.clean(ax, "correlation with churn, top 15 features", "", "pearson r (label 1 = churned)")
ax.grid(axis="y", visible=False)
plt.tight_layout()
corr.head(15).round(3)

everything that says "used the card a lot" is negative: log transactions −0.57, log counterparts −0.56, log spend −0.54, tenure −0.54, number of categories −0.54, countries −0.52. no single category share correlates much on its own. churners are customers who never really started, not customers who changed behaviour.

`leak_share` alone is flat (44 → 70 % only shows up once tenure is held fixed, see issue #9). ladder for the record: 0 % leakage 29 % churn, 5–20 % 44 %, 20–50 % 54 %, >50 % 70 %, among customers active > 1 year.

## rule baselines

what a one-line rule gets, so the models have something to beat.

In [ ]:
rules = {
    "always churned": np.ones_like(y),
    "days_active < 365": (train["days_active"] < 365).astype(int).to_numpy(),
    "n_transactions < 40": (train["n_transactions"] < 40).astype(int).to_numpy(),
}
pd.Series({k: accuracy_score(y, v) for k, v in rules.items()}, name="accuracy").round(3)

## three models, three feature sets, identical folds

- `raw`: the columns the course gave us (39 after cleaning)
- `engineered`: raw + ratios, category shares, leakage, international, logs, tenure bucket (76)
- `no_tenure`: engineered without `days_active` and everything derived from it (70). tells us how much of the label is tenure

repeated stratified 5-fold, 3 repeats, same seed for every model. metrics at threshold 0.5, then the accuracy-optimal threshold on the out-of-fold probabilities.

In [ ]:
models = {
    "logreg": make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=3000)),
    "random_forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=3, random_state=0, n_jobs=-1),
    "hist_gb": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=4, random_state=0),
}

rows, oof = [], {}
for fs in FEATURE_SETS:
    X_fs = feature_matrix(train, fs)
    for name, model in models.items():
        p, metrics = cross_validate(model, X_fs, y, n_splits=5, n_repeats=3, seed=0)
        thr, acc_thr = best_threshold(y, p)
        oof[(name, fs)] = p
        rows.append(summary_row(name, fs, metrics, thr, acc_thr))

results = pd.DataFrame(rows).sort_values(["feature_set", "accuracy"], ascending=[True, False]).reset_index(drop=True)
results[["model", "feature_set", "accuracy", "accuracy_std", "auc", "log_loss", "brier", "threshold", "accuracy_at_threshold"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=False)
for ax, metric, lim in zip(axes, ["accuracy", "auc"], [(0.70, 0.80), (0.80, 0.87)]):
    piv = results.pivot(index="model", columns="feature_set", values=metric)[list(FEATURE_SETS)]
    x = np.arange(len(piv))
    for i, fs in enumerate(piv.columns):
        ax.bar(x + (i - 1) * 0.26, piv[fs], width=0.24, color=style.C[i], label=fs)
    ax.set_xticks(x, piv.index)
    ax.set_ylim(*lim)
    style.clean(ax, f"{metric} by model and feature set (5-fold × 3)", metric)
axes[0].legend(loc="upper left", ncol=3, fontsize=8)
plt.tight_layout()

all three models sit at 0.76–0.77 accuracy and 0.85 AUC on the engineered set. logistic regression is as good as the tree models, which says the signal is mostly monotone "more usage → less churn".

the `no_tenure` set loses almost nothing: logreg 0.769 → 0.769 accuracy, AUC 0.854 → 0.843. so the label is not "days since last transaction" in disguise. usage volume (transactions, counterparts, spend) carries the same information as tenure, they're collinear. good to know before we argue with Lukas about what "churned" means.

`raw` vs `engineered`: +1 point accuracy for logreg, nothing for the trees. the trees find the ratios themselves.

## threshold

accuracy is what gets scored, so the cut-off is chosen on out-of-fold probabilities, not left at 0.5.

In [ ]:
best = results[results["feature_set"] == "engineered"].iloc[0]
best_name = best["model"]
p_oof = oof[(best_name, "engineered")]

grid = np.round(np.arange(0.30, 0.701, 0.005), 3)
acc_curve = [accuracy_score(y, p_oof >= t) for t in grid]
thr, acc_at_thr = best_threshold(y, p_oof)

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(grid, acc_curve, color=style.C[0], lw=2)
ax.axvline(thr, color=style.GREY, lw=1, ls="--")
ax.annotate(f"{thr:.3f} → {acc_at_thr:.3f}", (thr, acc_at_thr), xytext=(8, -14), textcoords="offset points", color=style.INK2)
style.clean(ax, f"{best_name}, engineered: out-of-fold accuracy vs threshold", "accuracy", "threshold")
plt.tight_layout()

## hand-in v1

fit the best model on all 3 903 labelled customers, predict the 1 673, write `results/churn_predictions.csv`. the writer checks row count, unique ids and that the ids equal `customer_data_predict.csv`.

In [ ]:
X_train, y_train, X_pred, pred_ids = train_predict_split(feats, "engineered")
final = models[best_name].fit(X_train, y_train)
p_pred = final.predict_proba(X_pred)[:, 1]

handin = write_handin(pred_ids, p_pred, thr)
print(f"{best_name} @ {thr:.3f}: predicted churn rate {handin['churned'].mean():.3f} (train rate {y.mean():.3f})")
handin.head()

In [ ]:
# results table and out-of-fold probabilities for issue #9 and the stacking in PR 2
save_results(rows)
oof_df = pd.DataFrame({"customer_id": train["customer_id"], "churned": y})
for (name, fs), p in oof.items():
    oof_df[f"oof_{name}_{fs}"] = np.round(p, 4)
oof_df.to_csv(ROOT / "results" / "oof_predictions_v1.csv", index=False)
print("written results/churn_models.csv and results/oof_predictions_v1.csv")

## next (PR 2)

- lightgbm, xgboost, catboost with optuna, stacking on the oof probabilities
- shap on the final model, dependence plots for `n_transactions`, `days_active`, `leak_share`, `foreign_tx_share`
- calibration (reliability diagram), the probabilities feed the value-at-risk in #10
- error analysis: who does the model get wrong, by tenure bucket
- cluster id as a feature once #7 exists
- ask Lukas: how is `churned` defined, when was it evaluated